# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a workflow for loading, exploring, and processing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and records from FAIR² using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant manifest URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset manifest/metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Title :', metadata.name)
print('Description:', metadata.description)

## 2. Data Overview

Review available record sets, fields, and their `@id`s. A record set in Croissant corresponds to a tabular dataset (like a table). Each record set contains fields (columns/variables), also referenced by `@id`.

**All entities are referenced by their `@id`.**

In [ ]:
# List all record sets available in the Croissant dataset schema
print('Record sets in this Croissant dataset:')
if not hasattr(metadata, 'record_set') or not metadata.record_set:
    print('⚠️ No record sets declared in the Croissant manifest. Inspect the dataset object for available entries.')
    # Try fetching record sets via dataset.list_record_sets()
    record_sets = dataset.list_record_sets()
else:
    record_sets = metadata.record_set
for i, rs in enumerate(record_sets):
    print(f"[{i}] @id: {rs['@id']} : name: {rs.get('name', '(unknown)')}")

# For illustration, we list fields for the first available RecordSet by @id.
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"\nFields for RecordSet @id {record_set_id}:")
    fields = dataset.list_fields(record_set=record_set_id)
    for x in fields:
        print(f"    field @id: {x['@id']}, name: {x.get('name',(x.get('@id')))}")

## 3. Data Extraction

Load the records for one or more record sets into pandas DataFrames for analysis.

*Use the record set and field `@id`s found above. Replace the record set IDs as needed to load additional tables.*

In [ ]:
# List all available record set @id's (adjust this list if needed)
record_set_ids = [r['@id'] for r in record_sets]

# Load each record set into pandas DataFrame by record set @id
dataframes = {}
for rs_id in record_set_ids:
    print(f'Loading record set: {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'  Columns: {df.columns.tolist()}')
        print(df.head(2))
    else:
        print(f'  [!] No records loaded for {rs_id}.')

# Example: preview columns in the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    if first_rs in dataframes:
        print('Available columns in DataFrame for', first_rs)
        print(dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply standard preprocessing and EDA steps: filter records, normalize numeric fields, categorize, and group. All columns are referenced by their `@id` as per the Croissant schema.

*Adjust the field @id's as appropriate to your use case and dataset content!*

In [ ]:
# --- Example EDA ---

# Use first available DataFrame/record set
if not dataframes:
    print('No data available for EDA.')
else:
    example_rs_id = list(dataframes.keys())[0]  # e.g., first record set
    df = dataframes[example_rs_id]

    print(f'Exploratory analysis on record set @id: {example_rs_id}')

    # List available (numeric) columns
    print('Sample numeric column candidates:')
    numeric_candidates = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidates.append(col)
            print(' ', col)
    if not numeric_candidates:
        print('No numeric columns detected.')
    else:
        # Pick the first numeric field for demo (replace with any field @id as needed)
        numeric_field_id = numeric_candidates[0]
        print(f'Using numeric field {numeric_field_id}.')

        # Filter rows with value greater than a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f'Filtered records with {numeric_field_id} > {threshold}:')
        display(filtered_df.head())

        # Z-score normalization
        normalized_field = f"{numeric_field_id}_normalized"
        filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f'Normalized {numeric_field_id} for filtered records:')
        display(filtered_df[[numeric_field_id, normalized_field]].head())

        # Try grouping by another field (choose first non-numeric field)
        group_candidates = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f'Grouping by {group_field_id}')
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped.head(10))
        else:
            print('No suitable categorical group field found for grouping analysis.')

## 5. Visualization

Visualize data using standard plots. Replace `numeric_field_id` and `group_field_id` with specific `@id` values as observed above for richer visualizations!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print('No dataframes to visualize.')
else:
    df = list(dataframes.values())[0]
    # Try auto-detecting numeric and categorical columns
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()
    if numeric_cols and cat_cols:
        group_field_id = cat_cols[0]
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore clinical data from the FAIR<sup>2</sup> dataset using Croissant schemas and the `mlcroissant` library. 

You can extend this workflow to perform advanced analyses, deeper statistics, and use the precise Croissant `@id` references in your own data science pipelines.